In [2]:

"""
FastQ Processor

This script implements:
- FASTQ file loading (manual parsing)
- Sliding window quality trimming
- Length filtering
- GC-content calculation
- Phred quality calculation
- Output of filtered reads
"""

def load_fastq(filename):
    """Load FASTQ file into a list of tuples: (header, sequence, quality)"""
    reads = []
    with open(filename) as f:
        while True:
            header = f.readline().strip()
            if not header:
                break
            seq = f.readline().strip()
            plus = f.readline().strip()
            qual = f.readline().strip()
            reads.append((header, seq, qual))
    return reads

def phred_score(char):
    """Convert ASCII character to Phred+33 score"""
    return ord(char) - 33

def sliding_window_trim(seq, qual, window=5, min_avg=30):
    """Trim read from 5' to 3' using sliding window"""
    for i in range(len(seq) - window + 1):
        window_scores = [phred_score(q) for q in qual[i:i+window]]
        avg = sum(window_scores) / window
        if avg < min_avg:
            return seq[:i], qual[:i]
    return seq, qual

def length_filter(seq, qual, min_len=1):
    """Return True if read length >= min_len"""
    return len(seq) >= min_len

def gc_content(seq):
    """Calculate GC content as percentage"""
    seq = seq.upper()
    gc = sum(1 for b in seq if b in "GC")
    atgc = sum(1 for b in seq if b in "ATGC")
    if atgc == 0:
        return 0
    return (gc / atgc) * 100

def average_phred_at_pos(reads, pos):
    """Compute average Phred score at a given 1-based position"""
    total, count = 0, 0
    idx = pos - 1
    for _, seq, qual in reads:
        if len(qual) > idx:
            total += phred_score(qual[idx])
            count += 1
    return total // count if count > 0 else 0

def process_fastq(input_file, output_file, window=5, min_q=30, min_len=60):
    """Process FASTQ file: trimming, filtering, stats, output"""
    reads = load_fastq(input_file)
    filtered = []
    
    removed_count = 0
    for h, seq, qual in reads:
        seq_trim, qual_trim = sliding_window_trim(seq, qual, window, min_q)
        if length_filter(seq_trim, qual_trim, min_len):
            filtered.append((h, seq_trim, qual_trim))
        else:
            removed_count += 1
    
    # Save filtered reads
    with open(output_file, "w") as f:
        for h, seq, qual in filtered:
            f.write(f"{h}\n{seq}\n+\n{qual}\n")
    
    # Statistics
    lengths = [len(seq) for _, seq, _ in filtered]
    min_len_out = min(lengths) if lengths else 0
    max_len_out = max(lengths) if lengths else 0
    avg_len_out = sum(lengths) // len(lengths) if lengths else 0
    
    gc_values = [gc_content(seq) for _, seq, _ in filtered]
    avg_gc = sum(gc_values) / len(gc_values) if gc_values else 0
    
    avg_phred10 = average_phred_at_pos(filtered, 10)
    
    print(f"Reads removed: {removed_count}")
    print(f"Filtered reads: {len(filtered)}")
    print(f"Length - min: {min_len_out}, avg: {avg_len_out}, max: {max_len_out}")
    print(f"Average GC content: {avg_gc:.2f}%")
    print(f"Average Phred at position 10: {avg_phred10}")

# Example usage
if __name__ == "__main__":
    process_fastq("reads.fastq", "filtered_reads.fastq", window=5, min_q=30, min_len=60)

Reads removed: 76558
Filtered reads: 162808
Length - min: 60, avg: 133, max: 151
Average GC content: 43.16%
Average Phred at position 10: 36
